### Products_data exploring

In [0]:
path="abfss://bronze@ecommercedatalakebp01.dfs.core.windows.net/products/ingestion_date=2026-09-23/products.csv"
df=spark.read.option("header",True).format("csv").load(path)
display(df)
print(df.count())
df.printSchema()

In [0]:
df.select("product_id").distinct().count()

In [0]:
df.filter(df.price.isNull()).count()

In [0]:
df.filter(df.product_id.isNull()).count()

In [0]:
df.filter(df.product_name.isNull()).count()

In [0]:
df.filter(df.category.isNull()).count()

In [0]:
df.filter(df.supplier_id.isNull()).count()

In [0]:
display(df.groupby("category").count())

In [0]:
from pyspark.sql import functions as F
df2=df.withColumn("price_num", F.col("price").try_cast("double"))
df2.display()

In [0]:
display(df2.filter((df2.price.isNotNull()) & (df2.price_num.isNull())))

In [0]:
display(df2.describe("price_num"))

In [0]:
df2.filter(df2.price_num<=0).count()

### returns_data exploring

In [0]:
path_r="abfss://bronze@ecommercedatalakebp01.dfs.core.windows.net/returns/ingestion_date=2026-09-23/returns.csv"
df_r=spark.read.format("csv").option("header","true").option("inferSchema","true").load(path_r)
df_r.display()
df_r.count()

In [0]:
df_r.select("return_id").distinct().count()

In [0]:
df_r.select("order_id").distinct().count()

In [0]:
df_r.select("return_date").distinct().count()

In [0]:
df_r.select("return_reason").distinct().count()

In [0]:
df_r.select("refund_amount").distinct().count()

In [0]:
df_r.filter(df_r.return_id.isNull()).count()
df_r.filter(df_r.order_id.isNull()).count()
df_r.filter(df_r.return_date.isNull()).count()
df_r.filter(df_r.return_reason.isNull()).count()
df_r.filter(df_r.refund_amount.isNull()).count()

In [0]:
df_r.describe("refund_amount").display()

In [0]:
df_r.printSchema()

In [0]:
df_r.groupBy("return_reason").count().display()

In [0]:
df_r.select(F.min("return_date"),F.max("return_date")).display()

In [0]:
df_r.filter(df_r.return_date > F.current_date()).count()


### Orders Exploring

In [0]:
path_o="abfss://bronze@ecommercedatalakebp01.dfs.core.windows.net/orders/ingestion_date=2026-09-23/orders.parquet"
df_o=spark.read.format("parquet").load(path_o)
df_o.printSchema()
df_o.display()

In [0]:
print(df_o.count())

In [0]:
df_o.select("order_id").distinct().count()

In [0]:
df_o1=df_o.groupBy("order_id").count()  

In [0]:
df_o1.filter(df_o1["count"]>1).display()

In [0]:
df_o.filter(df_o.order_id.isin(["O0000101", "O0000201", "O0000301"])).orderBy("order_id").display()

In [0]:
df_o.filter(df_o.order_id.isNull()).count()

In [0]:
df_o.filter(df_o.customer_id.isNull()).count()

In [0]:
df_o.filter(df_o.product_id.isNull()).count()

In [0]:
df_o.groupBy("order_status").count().display()

In [0]:
df_o.describe("quantity").display()
df_o.describe("price").display()
df_o.describe("revenue").display()

In [0]:
df_o.filter(df_o.quantity==0).count()

In [0]:
df_o2=df_o.withColumn("order_date_casted",F.col("order_date").try_cast("date"))

In [0]:
df_o2.filter((df_o2.order_date.isNotNull()) & (df_o2.order_date_casted.isNull())).count()

In [0]:
df_o2.select(F.min("order_date"),F.max("order_date")).display()
df_o2.filter(df_o2.order_date> F.current_date()).count()


### Customers data exploring

In [0]:
path_c="abfss://bronze@ecommercedatalakebp01.dfs.core.windows.net/customers/ingestion_date=2026-09-23/customers.parquet"
df_c=spark.read.format("parquet").load(path_c)
df_c.printSchema()
print(df_c.count())
df_c.display()

In [0]:
print(df_c.filter(df_c.customer_name.isNull()).count())
print(df_c.filter(df_c.email.isNull()).count())
print(df_c.filter(df_c.city.isNull()).count())
print(df_c.filter(df_c.state.isNull()).count())
print(df_c.filter(df_c.country.isNull()).count())
print(df_c.filter(df_c.signup_date.isNull()).count())


In [0]:
df_c.select("customer_id").distinct().count()

In [0]:
display(df_c.groupBy("city").count())

In [0]:
df_c.groupBy("state").count().orderBy("state").display()

In [0]:
df_c.groupBy("state", "city").count().orderBy("state", "city").display()


In [0]:
from pyspark.sql.functions import col
df_c2=df_c.withColumn("sign_up_date_parsed", F.try_to_date(col("signup_date"), "yyyy-MM-dd"))
df_c2.display()

In [0]:
print(df_c2.filter(df_c2.sign_up_date_parsed.isNull()).count())


In [0]:
df_c2.select(F.min("sign_up_date_parsed"),F.max("sign_up_date_parsed")).display()

In [0]:
df_c2.groupBy("country").count().display()

In [0]:
bad_emails = df_c2.filter(
    (F.col("email").isNotNull()) &
    (~F.col("email").contains("@"))
)

print("bad emails:", bad_emails.count())

In [0]:
orphans = df_o.join(df_c2, on="customer_id", how="left_anti")

print("orphan orders:", orphans.count())
orphans.select("order_id", "customer_id").display()

### Payments exploring

In [0]:
path_p="abfss://bronze@ecommercedatalakebp01.dfs.core.windows.net/payments/ingestion_date=2026-09-23/payments.json"
df_p = spark.read.json(path_p)
df_p.display()
df_p.printSchema()
df_p.count()


In [0]:
null_counts=df_p.select([F.count(F.when(F.col(c).isNull(), c)).alias(c) for c in df_p.columns])
null_counts.display()

In [0]:
df_p.select(df_p.payment_id).distinct().count()

In [0]:
df_p.select(df_p.order_id).distinct().count()

In [0]:
df_p.join(df_o,df_o["order_id"]==df_p["order_id"],"left_anti").display()

In [0]:
df_p1=df_p.withColumn("payment_date_parsed", F.try_to_date(col("payment_date"), "yyyy-MM-dd"))
df_p1.display()

In [0]:
print(df_p1.filter(df_p1.payment_date_parsed.isNull()).count())


In [0]:
display(df_p1.filter(df_p1.payment_date_parsed>F.current_date()).count())

In [0]:
df_p1.select(F.min("payment_date_parsed"),F.max("payment_date_parsed")).display()

In [0]:
display(df_p.groupBy("payment_method").count().orderBy("payment_method"))

In [0]:
display(df_p.groupBy("payment_status").count().orderBy("payment_status"))

In [0]:
df_p.describe("payment_amount").display()

In [0]:
df_p2=df_p.filter(df_p.payment_amount <0)
df_p2.count()
df_p2.display()


In [0]:
display(df_p2.groupBy("payment_status").count())

### Website events exploring

In [0]:
path_w="abfss://bronze@ecommercedatalakebp01.dfs.core.windows.net/website_events/ingestion_date=2026-09-23/website_events.csv"
df_w=spark.read.format("csv").option("header","true").load(path_w)
df_w.display()
df_w.printSchema()
df_w.count()

In [0]:
null_website_counts=df_w.select([F.count(F.when(F.col(c).isNull(), c)).alias(c) for c in df_w.columns])
null_website_counts.display()

In [0]:
df_w.select("event_id").distinct().count()

In [0]:
display(df_w.groupby("event_type").count().orderBy("event_type"))

In [0]:
display(df_w.filter(df_w.event_type=="unknown_event"))

In [0]:
df_w1=df_w.withColumn("event_timestamp_parsed", F.try_to_timestamp(F.col("event_timestamp"),F.lit("yyyy-MM-dd HH:mm:ss")))
df_w1.display()

In [0]:
print("unparsed timestamps:", df_w1.filter(F.col("event_timestamp_parsed").isNull()).count())
print("future events:", df_w1.filter(F.col("event_timestamp_parsed") > F.current_timestamp()).count())

In [0]:
display(df_w1.select(F.min("event_timestamp_parsed"),F.max("event_timestamp_parsed")))

### Cross checks

In [0]:
df_o.join(df,df_o["product_id"]==df["product_id"],"left_anti").display()

In [0]:
df_r.join(df_o,df_r["order_id"]==df_o["order_id"],"left_anti").display()